# Phase 6 - Abstractive Summarization (lightweight Phase-1 model)

**Task:** generate a short abstractive summary of a long legal document. Primary metric BERTScore F1 (computed by Codabench); secondary ROUGE-1 / ROUGE-L F1.

**What you need on disk (nothing else - the model code is embedded below):**
1. your Phase-1 lightweight checkpoint `lightweight_final.pt` (must contain `model_state` and `config`)
2. the Phase-1 tokenizer pickle `legal_bpe_v3.pkl` (the `.pt` does **not** contain it)
3. a Hugging Face token (Kaggle secret `hf_token`)

**Approach:** the lightweight LM (4 shared blocks run twice, causal 128-token local attention with ALiBi, grouped-query) is used as a prompt -> summary language model. Each example is `[document tokens] [separator "Summary:"] [summary tokens] <eos>`, trained with teacher forcing and the loss on the summary tokens only. The context is limited, so a long document is cut to the part that fits (`SELECT`: `head`, `headtail` = beginning + end, or `importance` = tf-idf sentence selection); a hierarchical chunk-then-summarise variant is available at inference time. Generation is greedy with a cache (a sliding window of keys/values per block pass - small, because the attention is local), a minimum length and a no-repeat n-gram rule (redundancy control).

**Experiments:** full fine-tuning, partial (layer-wise), LoRA, **LoRMA+** (compulsory), Houlsby adapters, Pfeiffer adapters, prompt tuning, knowledge distillation (student = half the unique blocks), then long-document, decoding-length and redundancy ablations. AdaLoRA, QLoRA and prefix tuning are **not** included.

> The course requires all **four** Phase-1 architectures and >= 6 substantially different experiments (>= 5 different adaptation techniques, LoRMA and knowledge distillation among them). The budgets (`minutes=`), learning rates and generation settings are placeholders - choosing and justifying them is your job (course rules). BERTScore needs a pretrained encoder, so it is **not** computed here: the local ROUGE numbers are only a proxy for choosing between runs. Memory-safe defaults (`bs=1, accum=16`, gradient checkpointing) are used because this model's windowed attention is memory-hungry.

## 0. Config - edit the paths

In [ ]:
TOKENIZER_PATH = "/kaggle/input/YOUR_DATASET/legal_bpe_v3.pkl"      # <-- Phase-1 tokenizer
CKPT_PATH      = "/kaggle/input/YOUR_DATASET/lightweight_final.pt"      # <-- Phase-1 lightweight checkpoint
HF_REPO   = "Exploration-Lab/CS-781-Capstone"
PHASE_DIR = "Phase_6_Summ"
OUT = "/kaggle/working"

# the data files are auto-detected from the repo listing; set these only if auto-detection fails
TRAIN_PATH = DEV_PATH = TEST_PATH = None
DEV_FRAC   = 0.05     # the Phase 6 repo has no dev file: this share of the training documents is held out as dev (seed 0)

GEN_MAX    = 200      # max summary length in tokens (training targets are cut to this; generation stops here)
N_RESERVE  = 32       # positions kept free for the soft prompt of the prompt-tuning experiment
SELECT     = "headtail"   # how a long document is cut to fit the context: head | headtail | importance
HEAD_FRAC  = 0.6      # headtail: fraction of the document budget taken from the beginning, the rest from the end
CAP_CHARS  = 12000    # characters read from each end of a document (bounds tokenisation time; importance reads more)

## 1. Imports and Hugging Face login

In [ ]:
import copy, json, math, os, pickle, random, re, time, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev, "| torch", torch.__version__)

from collections import Counter
from torch.utils.checkpoint import checkpoint

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("hf_token")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")        # Colab / local: export HF_TOKEN yourself
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN); print("HF login ok")
else:
    print("no HF token found - dataset download may fail")

## 2. Phase-1 tokenizer (class copied from your Phase-1 notebook)

In [ ]:
class LegalBPETokenizer:
    """Phase-1 tokenizer, copied unchanged from your Phase-1 notebook (needs legal_bpe_v3.pkl)."""

    def __init__(self, model_path):
        with open(model_path, "rb") as f:
            model = pickle.load(f)
        self.vocab_size = model["vocab_size"]
        self.merges = model["merges"]                     # [((a, b), new_id), ...]
        self.lowercase = model.get("lowercase", False)
        self.special_tokens = model.get("special_tokens", [])
        self.pattern = re.compile(model["pattern"])
        self.merge_rank = {tuple(pair): rank for rank, (pair, _) in enumerate(self.merges)}
        self.token_bytes = {i: bytes([i]) for i in range(256)}
        for (l, r), new_id in self.merges:
            self.token_bytes[new_id] = self.token_bytes[l] + self.token_bytes[r]
        base = 256 + len(self.merges)
        self.special_to_id = {t: base + i for i, t in enumerate(self.special_tokens)}
        self.id_to_special = {v: k for k, v in self.special_to_id.items()}
        self.eos_id = self.special_to_id.get("<|endoftext|>")
        self._special_re = (re.compile("(" + "|".join(re.escape(t) for t in self.special_tokens) + ")")
                            if self.special_tokens else None)
        self._cache = {}

    def _encode_piece(self, piece_bytes):
        cached = self._cache.get(piece_bytes)
        if cached is not None:
            return cached
        ids = list(piece_bytes)
        while len(ids) > 1:
            best, best_rank = None, None
            for p in zip(ids, ids[1:]):
                r = self.merge_rank.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = p, r
            if best is None:
                break
            new_id = 256 + best_rank
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == best[0] and ids[i + 1] == best[1]:
                    out.append(new_id); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        if len(self._cache) < 500_000:
            self._cache[piece_bytes] = ids
        return ids

    def _encode_ordinary(self, text):
        if self.lowercase:
            text = text.lower()
        out = []
        for piece in self.pattern.findall(text):
            out.extend(self._encode_piece(piece.encode("utf-8")))
        return out

    def encode(self, text, add_eos=False):
        if self._special_re is not None:
            ids = []
            for part in self._special_re.split(text):
                if part in self.special_to_id:
                    ids.append(self.special_to_id[part])
                elif part:
                    ids.extend(self._encode_ordinary(part))
        else:
            ids = self._encode_ordinary(text)
        if add_eos:
            ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        chunks = []
        for token_id in ids:
            if token_id in self.id_to_special:
                chunks.append(self.id_to_special[token_id].encode("utf-8"))
            else:
                chunks.append(self.token_bytes[token_id])
        return b"".join(chunks).decode("utf-8", errors="replace")


tokenizer = LegalBPETokenizer(TOKENIZER_PATH)
print("vocab size:", tokenizer.vocab_size, "| lowercase:", tokenizer.lowercase, "| eos id:", tokenizer.eos_id)
print([tokenizer.decode([i]) for i in tokenizer.encode("The appellant filed this appeal.")])

## 3. Phase-1 lightweight model code (copied unchanged) and checkpoint loading

In [ ]:
# ---- lightweight_layers.py (Phase 1, unchanged) ----
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        rms = x.float().pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(rms + self.eps)
        return x * self.weight.to(x.dtype)


def get_alibi_slopes(n_heads):
    def power_of_2_slopes(n):
        start = 2 ** (-2 ** -(math.log2(n) - 3))
        ratio = start
        return torch.tensor([start * ratio ** i for i in range(n)], dtype=torch.float32)
    if math.log2(n_heads).is_integer():
        return power_of_2_slopes(n_heads)
    closest_power = 2 ** int(math.floor(math.log2(n_heads)))
    extra = n_heads - closest_power
    slopes = power_of_2_slopes(closest_power)
    extra_slopes = get_alibi_slopes(2 * closest_power)[0::2]
    return torch.cat([slopes, extra_slopes[:extra]])


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.w3(F.silu(self.w1(x)) * self.w2(x)))


class LocalGQAAttention(nn.Module):
    """Causal sliding-window grouped-query attention with ALiBi."""
    def __init__(self, d_model, n_q_heads, n_kv_heads, max_seq_len, window_size, dropout, qk_norm=True):
        super().__init__()
        assert d_model % n_q_heads == 0 and n_q_heads % n_kv_heads == 0
        self.d_model = d_model
        self.n_q_heads, self.n_kv_heads = n_q_heads, n_kv_heads
        self.groups = n_q_heads // n_kv_heads
        self.d_head = d_model // n_q_heads
        self.window_size = min(window_size, max_seq_len)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.qk_norm = qk_norm
        if qk_norm:
            self.q_norm = RMSNorm(self.d_head)
            self.k_norm = RMSNorm(self.d_head)
        self.dropout = dropout
        self.register_buffer("alibi_slopes", get_alibi_slopes(n_q_heads), persistent=False)
        distance = torch.arange(self.window_size - 1, -1, -1, dtype=torch.float32)
        self.register_buffer("distance", distance, persistent=False)
        query_pos = torch.arange(max_seq_len).unsqueeze(1)
        window_pos = torch.arange(self.window_size).unsqueeze(0)
        valid = window_pos >= (self.window_size - 1 - query_pos)
        self.register_buffer("valid_mask", valid, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_q_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        if self.qk_norm:
            q, k = self.q_norm(q), self.k_norm(k)
        q = q.view(B, self.n_kv_heads, self.groups, T, self.d_head)
        W = self.window_size
        k_win = F.pad(k, (0, 0, W - 1, 0)).unfold(2, W, 1).permute(0, 1, 2, 4, 3)   # (B, Hkv, T, W, D)
        v_win = F.pad(v, (0, 0, W - 1, 0)).unfold(2, W, 1).permute(0, 1, 2, 4, 3)
        scores = torch.einsum("bhgtd,bhtwd->bhgtw", q, k_win) / math.sqrt(self.d_head)
        slopes = self.alibi_slopes.view(1, self.n_kv_heads, self.groups, 1, 1)
        scores = scores - slopes * self.distance.view(1, 1, 1, 1, W)
        mask = self.valid_mask[:T, :W].view(1, 1, 1, T, W)
        scores = scores.masked_fill(~mask, float("-inf"))
        attn = F.softmax(scores.float(), dim=-1).to(scores.dtype)
        if self.training and self.dropout > 0:
            attn = F.dropout(attn, p=self.dropout)
        out = torch.einsum("bhgtw,bhtwd->bhgtd", attn, v_win)
        out = out.reshape(B, self.n_q_heads, T, self.d_head).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class LightweightBlock(nn.Module):
    def __init__(self, d_model, n_q_heads, n_kv_heads, max_seq_len, window_size, d_ff, dropout, qk_norm):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.attn = LocalGQAAttention(d_model, n_q_heads, n_kv_heads, max_seq_len, window_size, dropout, qk_norm)
        self.ln2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.ffn(self.ln2(x))


# ---- model_arch_lightweight.py (Phase 1, unchanged) ----
from dataclasses import dataclass

@dataclass
class LightweightConfig:
    vocab_size: int
    d_model: int = 1024
    n_q_heads: int = 16
    n_kv_heads: int = 4
    d_ff: int = 4096
    max_seq_len: int = 1024
    window_size: int = 128
    dropout: float = 0.0
    qk_norm: bool = True
    n_unique_blocks: int = 4       # 4 unique blocks ...
    n_repeats: int = 2             # ... each executed twice: B1 B2 B3 B4 B1 B2 B3 B4


class LightweightLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([
            LightweightBlock(cfg.d_model, cfg.n_q_heads, cfg.n_kv_heads, cfg.max_seq_len, cfg.window_size,
                             cfg.d_ff, cfg.dropout, cfg.qk_norm)
            for _ in range(cfg.n_unique_blocks)])
        self.ln_f = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight            # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):
        x = self.tok_emb(x)
        for _ in range(self.cfg.n_repeats):
            for block in self.blocks:
                x = block(x)
        return self.lm_head(self.ln_f(x))

In [ ]:
def load_lm(path=CKPT_PATH, dropout=0.1):
    # weights_only=False: this is YOUR OWN checkpoint (config dict + tensors), not an untrusted file
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = LightweightConfig(**{k: v for k, v in ck["config"].items() if k in LightweightConfig.__dataclass_fields__})
    assert cfg.vocab_size == tokenizer.vocab_size, \
        f"checkpoint vocab {cfg.vocab_size} != tokenizer vocab {tokenizer.vocab_size}: wrong tokenizer .pkl?"
    cfg.dropout = dropout                                    # Phase 1 used 0.0; a little helps fine-tuning
    lm = LightweightLM(cfg)
    lm.load_state_dict(ck["model_state"])                    # alibi / mask buffers are non-persistent: rebuilt, not loaded
    return lm

_lm = load_lm(dropout=0.0)
print(_lm.cfg)
print(f"loaded Phase-1 lightweight LM: {sum(p.numel() for p in _lm.parameters())/1e6:.1f}M unique params "
      f"({_lm.cfg.n_unique_blocks} blocks x {_lm.cfg.n_repeats} repeats)")
# (the context length is read from the checkpoint in section 4.3)

## 4. Data
### 4.1 Find the files and peek at the raw JSON (verify the layout and the keys before trusting the parser)

In [ ]:
from huggingface_hub import list_repo_files
files = sorted(f for f in list_repo_files(HF_REPO, repo_type="dataset") if f.startswith(PHASE_DIR))
print("\n".join(files))

def pick(kind, override, required=True):
    if override: return override
    c = [f for f in files if f.endswith(".json") and kind in f.split("/")[-1].lower()]
    if kind == "dev" and not c:
        c = [f for f in files if f.endswith(".json") and "val" in f.split("/")[-1].lower()]
    if not c and not required:
        return None
    assert len(c) == 1, f"could not pick the {kind} file from {files} - set {kind.upper()}_PATH in the config cell"
    return c[0]

TRAIN_PATH, DEV_PATH, TEST_PATH = pick("train", TRAIN_PATH), pick("dev", DEV_PATH, required=False), pick("test", TEST_PATH)
print("\nusing:", TRAIN_PATH, DEV_PATH or "(no dev file: a dev split is held out of the training data)", TEST_PATH)

def hf_json(path):
    return json.load(open(hf_hub_download(HF_REPO, path, repo_type="dataset"), encoding="utf-8"))

for name, path in [("train", TRAIN_PATH), ("dev", DEV_PATH), ("test", TEST_PATH)]:
    if path is None: continue
    raw = hf_json(path)
    first = next(iter(raw.items())) if isinstance(raw, dict) else raw[0]
    if isinstance(first, dict):
        show = {k: (str(v)[:150] + "..." if len(str(v)) > 150 else v) for k, v in first.items()}
    else:
        show = str(first)[:300]
    print(name, type(raw).__name__, "len", len(raw), "\n   first:", show, "\n")

### 4.2 Parse train / dev / test

In [ ]:
DOC_KEYS = ("document", "text", "judgment", "doc", "input", "source")
SUM_KEYS = ("summary", "abstract", "headnote", "target", "reference", "gold", "output")

def as_text(v):
    if isinstance(v, (list, tuple)):
        return " ".join(str(x) for x in v)
    return v if isinstance(v, str) else (None if v is None else str(v))

def detect_keys(rec):
    """Which keys hold the document and the summary (by the usual names; otherwise: the longest text field is the document,
    the other text field is the summary)."""
    txt = [k for k, v in rec.items() if k != "id" and isinstance(v, (str, list, tuple))]
    dk = next((k for k in DOC_KEYS if k in rec), None)
    sk = next((k for k in SUM_KEYS if k in rec and k != dk), None)
    if dk is None and txt:
        dk = max(txt, key=lambda k: len(as_text(rec[k]) or ""))
    if sk is None:
        rest = [k for k in txt if k != dk]
        sk = rest[0] if len(rest) == 1 else None
    return dk, sk

def parse_docs(raw, keys=None):
    items = list(raw.items()) if isinstance(raw, dict) else [(d.get("id", i), d) for i, d in enumerate(raw)]
    docs = []
    for did, d in items:
        if not isinstance(d, dict):
            d = {"document": d}
        dk, sk = keys if keys else detect_keys(d)
        docs.append(dict(id=d.get("id", did), text=as_text(d.get(dk)) or "", summary=as_text(d.get(sk)) if sk else None))
    return docs

_raw_train = hf_json(TRAIN_PATH)
_first = next(iter(_raw_train.values())) if isinstance(_raw_train, dict) else _raw_train[0]
KEYS = detect_keys(_first)
print("record keys:", list(_first.keys()) if isinstance(_first, dict) else type(_first), "-> document key:", KEYS[0], "| summary key:", KEYS[1])
assert KEYS[0] and KEYS[1], "could not tell which keys hold the document and the summary - set KEYS = (doc_key, summary_key) by hand"

train_docs = parse_docs(_raw_train, KEYS)
test_docs  = parse_docs(hf_json(TEST_PATH), (KEYS[0], None))
if DEV_PATH:
    dev_docs = parse_docs(hf_json(DEV_PATH), KEYS)
else:
    idx = list(range(len(train_docs))); random.Random(0).shuffle(idx)
    n_dev = min(max(100, int(DEV_FRAC * len(train_docs))), len(train_docs) // 4)
    held = set(idx[:n_dev])
    dev_docs = [train_docs[i] for i in idx[:n_dev]]
    train_docs = [d for i, d in enumerate(train_docs) if i not in held]
    print(f"no dev file in the repo: held out {n_dev} training documents as dev (fixed seed)")
_n0 = len(train_docs) + len(dev_docs)
train_docs = [d for d in train_docs if d["summary"] and d["summary"].strip() and d["text"].strip()]
dev_docs   = [d for d in dev_docs if d["summary"] and d["summary"].strip() and d["text"].strip()]
if len(train_docs) + len(dev_docs) < _n0:
    print(f"dropped {_n0 - len(train_docs) - len(dev_docs)} documents with an empty document or summary")
assert train_docs and dev_docs, "no training documents with a summary - check the record keys printed above"
assert len({str(d["id"]) for d in test_docs}) == len(test_docs), "duplicate test ids"
for name, ds in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:
    L = [len(d["text"]) for d in ds]
    extra = f" | summary words: median {int(np.median([len(d['summary'].split()) for d in ds]))}, p90 {int(np.percentile([len(d['summary'].split()) for d in ds], 90))}" if ds[0]["summary"] else ""
    print(f"{name}: {len(ds)} docs | document chars: median {int(np.median(L))}, p90 {int(np.percentile(L, 90))}, max {max(L)}{extra}")

### 4.3 Prompt building: how a long document is cut to fit the context (head | headtail | importance)

In [ ]:
SEP_IDS = tokenizer.encode("\n\nSummary:\n")
EOS = tokenizer.eos_id
CTX = _lm.cfg.max_seq_len                                        # decoder context length (from the checkpoint)
DOC_BUDGET = CTX - GEN_MAX - len(SEP_IDS) - N_RESERVE            # tokens of the document that fit in the prompt
assert DOC_BUDGET >= 128, f"context {CTX} too small for GEN_MAX={GEN_MAX}: lower GEN_MAX"
print(f"decoder context {CTX} = document {DOC_BUDGET} + separator {len(SEP_IDS)} + summary {GEN_MAX} + {N_RESERVE} reserved")

WORD = re.compile(r"[a-z]{3,}")
SENT = re.compile(r"(?<=[.!?])\s+|\n{2,}")

def split_sents(text):
    return [s.strip() for s in SENT.split(text) if len(s.strip()) > 20]

# document frequencies from the training documents (importance-based selection)
df = defaultdict(int)
for d in train_docs:
    for w in set(WORD.findall(d["text"][:200000].lower())):
        df[w] += 1
IDF = {w: math.log((1 + len(train_docs)) / (1 + c)) for w, c in df.items()}
IDF_DEFAULT = math.log(1 + len(train_docs))

_sent_cache = {}
def enc_sent(s):
    r = _sent_cache.get(s)
    if r is None:
        r = _sent_cache[s] = tokenizer.encode(" " + s)
    return r

def select_head(text, budget):
    return tokenizer.encode(text[:max(6 * budget, 2000)])[:budget]

def select_headtail(text, budget):
    if len(text) <= 2 * CAP_CHARS:
        ids = tokenizer.encode(text)
    else:
        ids = tokenizer.encode(text[:CAP_CHARS]) + tokenizer.encode(" ... ") + tokenizer.encode(text[-CAP_CHARS:])
    if len(ids) <= budget:
        return ids
    h = int(HEAD_FRAC * budget)
    return ids[:h] + ids[-(budget - h):]

def select_importance(text, budget):
    """Extractive pre-selection: score each sentence by the tf-idf mass of its words (first/last sentences get a bonus),
    keep the best sentences that fit in the budget, and restore the original order."""
    sents = split_sents(text[:400000])
    if not sents:
        return select_head(text, budget)
    words = [WORD.findall(s.lower()) for s in sents]
    tf = defaultdict(int)
    for ws in words:
        for w in ws: tf[w] += 1
    score = []
    for i, ws in enumerate(words):
        sc = sum(tf[w] * IDF.get(w, IDF_DEFAULT) for w in set(ws)) / max(8, len(ws))
        if i < 3 or i >= len(sents) - 3: sc *= 1.5
        score.append(sc)
    chosen, used = [], 0
    for i in sorted(range(len(sents)), key=lambda i: -score[i]):
        n = len(enc_sent(sents[i]))
        if used + n <= budget:
            chosen.append(i); used += n
        if used >= budget - 8:
            break
    out = []
    for i in sorted(chosen):
        out += enc_sent(sents[i])
    return out[:budget]

SELECTORS = {"head": select_head, "headtail": select_headtail, "importance": select_importance}

def get_prompt(d, select=None, budget=None):
    """Prompt token ids = selected part of the document + separator (cached per document and strategy)."""
    select = select or SELECT
    cache = d.setdefault("_p", {})
    key = (select, budget)
    if key not in cache:
        cache[key] = SELECTORS[select](d["text"], budget or DOC_BUDGET) + SEP_IDS
    return cache[key]

def summary_ids(d):
    if "_s" not in d:
        d["_s"] = (tokenizer.encode(d["summary"])[:GEN_MAX - 1]) + [EOS]
    return d["_s"]

t0 = time.time()
for d in train_docs[:50]:
    get_prompt(d); summary_ids(d)
print(f"prompt building: {(time.time()-t0)/50*1000:.0f} ms/doc (head-tail) | example prompt tokens: {len(get_prompt(train_docs[0]))}")
print("summary tokens: median", int(np.median([len(summary_ids(d)) for d in train_docs[:200]])),
      "| p90", int(np.percentile([len(summary_ids(d)) for d in train_docs[:200]], 90)), "| cap", GEN_MAX)
print("--- example reference summary ---\n", train_docs[0]["summary"][:600])

### 4.4 Checkpoint / tokenizer sanity check

In [ ]:
# Does checkpoint + tokenizer agree? The LM loss on real legal text should be ~3-4 nats; ~10 means a wrong tokenizer pkl.
x = torch.tensor([get_prompt(train_docs[0])[:512]])
with torch.no_grad():
    lg = _lm(x)
print("Phase-1 LM loss on the start of a training document:", round(F.cross_entropy(lg[0, :-1], x[0, 1:]).item(), 3))
del _lm

## 5. Metrics (local ROUGE-1/2/L, repetition, length ratio)

In [ ]:
def _words(s):
    return re.findall(r"[a-z0-9]+", s.lower())

def _f1(overlap, n_pred, n_ref):
    p = overlap / n_pred if n_pred else 0.0; r = overlap / n_ref if n_ref else 0.0
    return 2 * p * r / (p + r) if p + r else 0.0

def _grams(w, n):
    c = defaultdict(int)
    for i in range(len(w) - n + 1): c[tuple(w[i:i + n])] += 1
    return c

def rouge_n(pred, ref, n):
    a, b = _grams(_words(pred), n), _grams(_words(ref), n)
    return _f1(sum(min(c, b[g]) for g, c in a.items() if g in b), sum(a.values()), sum(b.values()))

def lcs_len(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]

def rouge_l(pred, ref):
    a, b = _words(pred)[:400], _words(ref)[:400]
    return _f1(lcs_len(a, b), len(a), len(b))

def rep_rate(text):
    """share of repeated word trigrams inside one summary (0 = no repetition)."""
    w = _words(text); g = [tuple(w[i:i + 3]) for i in range(len(w) - 2)]
    return 1 - len(set(g)) / len(g) if g else 0.0

def summary_scores(preds, refs):
    """Local proxies for the secondary metrics (the official scorer may use stemming / other tokenisation, so expect small differences).
    The primary metric, BERTScore F1, is computed by Codabench."""
    n = len(preds)
    return dict(rouge1=sum(rouge_n(p, r, 1) for p, r in zip(preds, refs)) / n,
                rouge2=sum(rouge_n(p, r, 2) for p, r in zip(preds, refs)) / n,
                rougeL=sum(rouge_l(p, r) for p, r in zip(preds, refs)) / n,
                rep=sum(rep_rate(p) for p in preds) / n,
                len_ratio=sum(len(_words(p)) for p in preds) / max(1, sum(len(_words(r)) for r in refs)))

## 6. Adaptation methods: LoRA, LoRMA+, Houlsby / Pfeiffer adapters
The 4 unique blocks are run twice, so their adapters are shared across both passes.

In [ ]:
class LoRALinear(nn.Module):
    """LoRA: W x + s * B A x with W frozen.  rs=True -> rank-stabilised scaling s = alpha / sqrt(r) (rsLoRA)."""
    def __init__(self, base, r=16, alpha=32, rs=False, dropout=0.05):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / math.sqrt(r) if rs else alpha / r
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.T @ self.B.T) * self.scale

def apply_lora(module, r=16, alpha=32, rs=False):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(module, name, LoRALinear(child, r, alpha, rs))
        else:
            apply_lora(child, r, alpha, rs)


class LoRMALinear(nn.Module):
    """LoRMA+ (Bihany, Patel, Modi 2025): multiplicative update  h = (I + (alpha/r) B A) W0 x ,  B=0 at init."""
    def __init__(self, base, r=8, alpha=8):
        super().__init__()
        self.base = base
        d = base.out_features
        self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.scale = alpha / r
    def forward(self, x):
        z = F.linear(x, self.base.weight)
        z = z + self.scale * ((z @ self.A.T) @ self.B.T)
        return z + self.base.bias if self.base.bias is not None else z

def apply_lorma(module, targets=("q_proj", "v_proj"), r=8, alpha=8):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, LoRMALinear(child, r, alpha))
        else:
            apply_lorma(child, targets, r, alpha)


class IA3Out(nn.Module):
    """IA3: learned vector rescales the layer OUTPUT (used for keys and values). Init = ones (identity)."""
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.out_features))
    def forward(self, x):
        return self.base(x) * self.l

class IA3In(nn.Module):
    """IA3 on the FFN: learned vector rescales the hidden activation that feeds the second FFN linear."""
    def __init__(self, base):
        super().__init__(); self.base = base; self.l = nn.Parameter(torch.ones(base.in_features))
    def forward(self, x):
        return self.base(x * self.l)

def apply_ia3(blocks):
    for b in blocks:
        b.attn.k_proj = IA3Out(b.attn.k_proj)
        b.attn.v_proj = IA3Out(b.attn.v_proj)
        b.ffn.net[2] = IA3In(b.ffn.net[2])

In [ ]:
class SubAdapter(nn.Module):
    """Bottleneck adapter on a sub-layer OUTPUT, before the residual add: y + up(GELU(down(y))). `up` starts at zero = identity.
    Houlsby = one after the attention AND one after the FFN of every block; Pfeiffer = only after the FFN."""
    def __init__(self, sub, d_model, bottleneck=64):
        super().__init__()
        self.sub = sub
        self.down = nn.Linear(d_model, bottleneck); self.up = nn.Linear(bottleneck, d_model)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def adapt(self, y):
        return y + self.up(F.gelu(self.down(y)))
    def forward(self, x):
        return self.adapt(self.sub(x))

def apply_adapters(lm, bottleneck=64, where="houlsby"):
    for b in lm.blocks:
        if where == "houlsby":
            b.attn = SubAdapter(b.attn, lm.cfg.d_model, bottleneck)
        b.ffn = SubAdapter(b.ffn, lm.cfg.d_model, bottleneck)

## 7. Model wrapper, `build_model`, distillation student, loss

In [ ]:
class SumLM(nn.Module):
    """Phase-1 lightweight LM (4 shared blocks x 2 repeats = 8 block passes, causal 128-token local attention) used as a
    prompt -> summary language model: document, separator, then the summary is generated. Optional soft prompt (prompt tuning).
    Training runs the whole [document ; summary] sequence once with teacher forcing; the LM loss is taken only on summary positions."""
    def __init__(self, lm, n_prompt=0):
        super().__init__()
        self.lm, self.n_prompt = lm, n_prompt
        d = lm.cfg.d_model
        self.prompt = nn.Parameter(torch.randn(n_prompt, d) * 0.02) if n_prompt else None
        self.n_nograd, self.use_ckpt = 0, False

    def passes(self):
        return [blk for _ in range(self.lm.cfg.n_repeats) for blk in self.lm.blocks]      # the unique blocks run n_repeats times

    def head_params(self):
        return [self.prompt] if self.prompt is not None else []

    def set_top_blocks(self, top):
        """Only the last `top` of the 8 block passes get gradients; the embedding and the earlier passes run under no_grad.
        The unique blocks are shared by all passes, so their weights still train through the later passes."""
        self.n_nograd = max(0, len(self.passes()) - top)
        for p in self.lm.tok_emb.parameters(): p.requires_grad = False

    def hidden(self, x):
        lm, nf, ps = self.lm, self.n_nograd, self.passes()
        def embed(x):
            h = lm.tok_emb(x)
            if self.prompt is not None:
                h = torch.cat([self.prompt.unsqueeze(0).expand(x.size(0), -1, -1).to(h.dtype), h], 1)
            return h
        if nf:
            with torch.no_grad():
                h = embed(x)
                for blk in ps[:nf]: h = blk(h)
        else:
            h = embed(x)
        for blk in ps[nf:]:
            h = checkpoint(blk, h, use_reentrant=False) if (self.use_ckpt and self.training and torch.is_grad_enabled()) else blk(h)
        return lm.ln_f(h)[:, self.n_prompt:]


def build_model(method, r=16, alpha=32, lorma_r=8, lorma_alpha=8, n_prompt=20, bottleneck=64):
    lm = load_lm()
    prompt = 0
    if method not in ("full", "partial"):
        for p in lm.parameters(): p.requires_grad = False
    if method == "lora":        apply_lora(lm.blocks, r, alpha, rs=False)
    elif method == "rslora":    apply_lora(lm.blocks, r, alpha, rs=True)
    elif method == "lorma":     apply_lorma(lm.blocks, ("q_proj", "v_proj"), lorma_r, lorma_alpha)
    elif method == "prompt":    prompt = n_prompt
    elif method == "houlsby":   apply_adapters(lm, bottleneck, "houlsby")
    elif method == "pfeiffer":  apply_adapters(lm, bottleneck, "pfeiffer")
    elif method not in ("full", "partial"):
        raise ValueError(method)
    m = SumLM(lm, n_prompt=prompt).to(dev)
    if method == "partial":
        m.set_top_blocks(len(m.passes()) // 2)       # layer-wise: embedding + the first repeat (4 of 8 passes) get no gradients
    return m


def make_student(teacher):
    """Smaller student: keep unique blocks 0 and 2 (half the block parameters) and still run them n_repeats times
    (depth 8 -> 4). Shared embeddings, initialised from the teacher. The teacher must be the plain full-fine-tuned model."""
    st = copy.deepcopy(teacher)
    keep = list(range(0, len(st.lm.blocks), 2))
    st.lm.blocks = nn.ModuleList([st.lm.blocks[i] for i in keep])
    st.lm.cfg.n_unique_blocks = len(keep)
    st.n_nograd = 0
    for p in st.parameters(): p.requires_grad = True
    return st


def collate(items):
    """items: list of (prompt_ids, summary_ids) -> X (B,T) right-padded inputs, L (B,T) labels (-100 except summary tokens)."""
    seqs = [p + s for p, s in items]; T = max(len(s) for s in seqs)
    X = torch.full((len(seqs), T), EOS, dtype=torch.long); L = torch.full((len(seqs), T), -100, dtype=torch.long)
    for i, ((p, s), seq) in enumerate(zip(items, seqs)):
        X[i, :len(seq)] = torch.tensor(seq); L[i, len(p):len(seq)] = torch.tensor(s)
    return X, L


def seq_loss(model, X, L, teacher=None, T=2.0, alpha=0.5):
    """Mean cross-entropy over summary tokens (+ temperature-scaled KL to the teacher's token distribution for KD).
    The vocabulary projection is applied only at summary positions (memory)."""
    h = model.hidden(X)[:, :-1]; lab = L[:, 1:]; m = lab != -100
    logits = model.lm.lm_head(h[m]).float()
    ce = F.cross_entropy(logits, lab[m])
    if teacher is None:
        return ce, ce
    with torch.no_grad():
        tl = teacher.lm.lm_head(teacher.hidden(X)[:, :-1][m]).float()
    kl = F.kl_div(F.log_softmax(logits / T, -1), F.softmax(tl / T, -1), reduction="batchmean") * T ** 2
    return alpha * ce + (1 - alpha) * kl, ce

## 8. Generation: cached greedy decoding (sliding key/value windows), no-repeat n-grams, hierarchical variant

In [ ]:
def _core(attn):
    return attn.sub if isinstance(attn, SubAdapter) else attn

def kv_of(core, r):
    """Keys / values the attention computes for the normed block input r (B,T,C): (B, Hkv, T, Dh) each (after QK-norm)."""
    B, T, _ = r.shape
    k = core.k_proj(r).view(B, T, core.n_kv_heads, core.d_head).transpose(1, 2)
    v = core.v_proj(r).view(B, T, core.n_kv_heads, core.d_head).transpose(1, 2)
    if core.qk_norm: k = core.k_norm(k)
    return k, v

def lw_prefill_block(blk, x):
    """The Phase-1 block on a whole prompt (exact original forward) + the last window-1 keys/values for the cache."""
    core = _core(blk.attn); W = core.window_size
    r = blk.ln1(x)
    k, v = kv_of(core, r)
    x = x + blk.attn(r)                                   # original (possibly adapter-wrapped) attention
    x = x + blk.ffn(blk.ln2(x))
    n = min(x.size(1), W - 1)
    return x, (k[:, :, x.size(1) - n:], v[:, :, x.size(1) - n:], n)

def lw_decode_block(blk, x, cache):
    """One new token per sequence. cache = (K, V, valid): the previous window-1 keys/values of this pass (B,Hkv,W-1,Dh), (B,W-1)."""
    attn = blk.attn; ad = attn if isinstance(attn, SubAdapter) else None; core = _core(attn)
    B, _, C = x.shape
    Hq, Hkv, Dh = core.n_q_heads, core.n_kv_heads, core.d_head; g = Hq // Hkv; W = core.window_size
    r = blk.ln1(x)
    q = core.q_proj(r).view(B, 1, Hq, Dh).transpose(1, 2)
    k, v = kv_of(core, r)
    if core.qk_norm: q = core.q_norm(q)
    K = torch.cat([cache[0], k.to(cache[0].dtype)], 2); V = torch.cat([cache[1], v.to(cache[1].dtype)], 2)      # (B,Hkv,W,Dh)
    valid = torch.cat([cache[2], torch.ones(B, 1, dtype=torch.bool, device=x.device)], 1)                       # (B,W)
    q = q.view(B, Hkv, g, 1, Dh)
    scores = torch.einsum("bhgtd,bhwd->bhgtw", q, K.to(q.dtype)) / math.sqrt(Dh)
    scores = scores - core.alibi_slopes.view(1, Hkv, g, 1, 1) * core.distance.view(1, 1, 1, 1, W)
    scores = scores.masked_fill(~valid.view(B, 1, 1, 1, W), float("-inf"))
    w = F.softmax(scores.float(), dim=-1).to(scores.dtype)
    o = torch.einsum("bhgtw,bhwd->bhgtd", w, V.to(w.dtype)).reshape(B, Hq, 1, Dh).transpose(1, 2).contiguous().view(B, 1, C)
    o = core.out_proj(o)
    if ad is not None: o = ad.adapt(o)
    x = x + o
    x = x + blk.ffn(blk.ln2(x))
    return x, (K[:, :, 1:], V[:, :, 1:], valid[:, 1:])


@torch.no_grad()
def generate_batch(model, prompts, max_new=None, min_new=30, nrep=3, debug_steps=0):
    """Greedy decoding with a cache for a batch of prompts. Every prompt is prefilled on its own (exact, no padding inside the model),
    the per-pass key/value windows are stacked (left-padded + validity mask) and the batch is decoded one token at a time.
    min_new: <eos> is forbidden for the first min_new tokens. nrep: no-repeat n-gram size (0 = off).
    Returns a list of generated token-id lists (without <eos>)."""
    model.eval(); lm = model.lm; max_new = max_new or GEN_MAX
    B, P = len(prompts), model.n_prompt
    ps = model.passes(); W1 = _core(ps[0].attn).window_size - 1
    res = []
    for p in prompts:
        x = lm.tok_emb(torch.tensor([p], device=dev))
        if P: x = torch.cat([model.prompt.unsqueeze(0).to(x.dtype), x], 1)
        sts = []
        for blk in ps:
            x, st = lw_prefill_block(blk, x); sts.append(st)
        res.append((lm.lm_head(lm.ln_f(x[:, -1])).float(), sts))
    caches = []
    for pi in range(len(ps)):
        K = torch.cat([F.pad(r[1][pi][0], (0, 0, W1 - r[1][pi][2], 0)) for r in res], 0)
        V = torch.cat([F.pad(r[1][pi][1], (0, 0, W1 - r[1][pi][2], 0)) for r in res], 0)
        valid = torch.zeros(B, W1, dtype=torch.bool, device=dev)
        for b, r in enumerate(res): valid[b, W1 - r[1][pi][2]:] = True
        caches.append((K, V, valid))
    logits = torch.cat([r[0] for r in res], 0)
    done = torch.zeros(B, dtype=torch.bool, device=dev)
    out = [[] for _ in range(B)]; dbg = []
    for step in range(max_new):
        if step < debug_steps: dbg.append(logits.clone())
        if step < min_new: logits[:, EOS] = -float("inf")
        if nrep and step >= nrep - 1:
            for i in range(B):
                if done[i]: continue
                g = out[i]
                if len(g) >= nrep - 1:
                    tail = tuple(g[len(g) - (nrep - 1):]) if nrep > 1 else ()
                    banned = {g[j + nrep - 1] for j in range(len(g) - nrep + 1) if tuple(g[j:j + nrep - 1]) == tail}
                    if banned: logits[i, list(banned)] = -float("inf")
        nxt = logits.argmax(-1)
        for i in range(B):
            if not done[i]:
                t = int(nxt[i])
                if t == EOS: done[i] = True
                else: out[i].append(t)
        if bool(done.all()) or step == max_new - 1:
            break
        x = lm.tok_emb(nxt.unsqueeze(1))
        for pi, blk in enumerate(ps):
            x, caches[pi] = lw_decode_block(blk, x, caches[pi])
        logits = lm.lm_head(lm.ln_f(x[:, 0])).float()
    return (out, dbg) if debug_steps else out


def check_cache(model):
    """The cached decoder must give the same next-token logits as a plain forward pass (prompts of different length, batched;
    one prompt is shorter than the attention window plus the generated tokens, so the window edge is exercised too)."""
    ps = [get_prompt(train_docs[0]), get_prompt(train_docs[1])[:300], get_prompt(train_docs[2])[:60]]
    with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
        out, dbg = generate_batch(model, ps, max_new=3, min_new=3, nrep=0, debug_steps=3)
        worst = 0.0
        for i, p in enumerate(ps):
            for s in range(3):
                ctx = p + out[i][:s]
                with torch.no_grad():
                    ref = model.lm.lm_head(model.hidden(torch.tensor([ctx], device=dev))[:, -1]).float()[0]
                err = (ref - dbg[s][i]).abs().max().item(); worst = max(worst, err)
    print(f"max |logit difference| cached vs full forward: {worst:.3f} (fp16; < ~0.5 is fine)")
    assert worst < 1.0, "cached decoding does not match the full forward pass"

def decode_ids(ids):
    return tokenizer.decode(ids).strip()


def summarise(model, docs, select=None, bs=8, max_new=None, min_new=30, nrep=3, prompts=None):
    """Summaries (strings) for `docs`, in order. Prompts can be passed in (used by the hierarchical variant)."""
    prompts = prompts or [get_prompt(d, select) for d in docs]
    order = sorted(range(len(docs)), key=lambda i: len(prompts[i]))
    outs = [None] * len(docs)
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
            gen = generate_batch(model, [prompts[j] for j in idx], max_new, min_new, nrep)
        for j, g in zip(idx, gen):
            outs[j] = decode_ids(g) or docs[j]["text"][:300]
    return outs


def summarise_hier(model, docs, n_chunks=5, chunk_new=80, **kw):
    """Hierarchical: split the document into up to n_chunks pieces that each fit the context, summarise every piece briefly,
    then summarise the concatenated chunk summaries. (The model is trained on single-pass prompts, so this is an inference-time ablation.)"""
    flat, owner = [], []
    for di, d in enumerate(docs):
        n = len(d["text"]); L = max(1, math.ceil(n / (4 * DOC_BUDGET))); L = min(L, n_chunks)
        step = n / L
        for c in range(L):
            piece = d["text"][int(c * step): int((c + 1) * step)]
            flat.append(tokenizer.encode(piece[:CAP_CHARS])[:DOC_BUDGET] + SEP_IDS); owner.append(di)
    stage1 = []
    for i in range(0, len(flat), 64):
        chunk = flat[i:i + 64]
        stage1 += summarise(model, [{"text": ""}] * len(chunk), prompts=chunk, max_new=chunk_new, min_new=10, nrep=kw.get("nrep", 3))
    merged = defaultdict(list)
    for o, s in zip(owner, stage1): merged[o].append(s)
    prompts2 = [tokenizer.encode(" ".join(merged[i]))[:DOC_BUDGET] + SEP_IDS for i in range(len(docs))]
    return summarise(model, docs, prompts=prompts2, **kw)

## 9. Evaluation helpers

In [ ]:
@torch.no_grad()
def dev_loss(model, docs, n=100, bs=1, select=None):
    """Teacher-forced cross-entropy on the reference summaries (fast training monitor; lower is better)."""
    model.eval(); tot, cnt = 0.0, 0
    for i in range(0, min(n, len(docs)), bs):
        batch = docs[i:min(n, len(docs))][:bs]
        X, L = collate([(get_prompt(d, select), summary_ids(d)) for d in batch])
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
            _, ce = seq_loss(model, X.to(dev), L.to(dev))
        k = int((L[:, 1:] != -100).sum()); tot += ce.item() * k; cnt += k
    return tot / max(1, cnt)

def dev_eval(model, docs, n=100, select=None, verbose=True, **gen_kw):
    sub = docs[:n]
    preds = summarise(model, sub, select, **gen_kw)
    sc = summary_scores(preds, [d["summary"] for d in sub])
    sc["preds"] = preds
    if verbose:
        print(f"   dev ({len(sub)} docs) ROUGE-1 {sc['rouge1']:.4f} ROUGE-2 {sc['rouge2']:.4f} ROUGE-L {sc['rougeL']:.4f} | "
              f"repeated-trigram share {sc['rep']:.3f} | length ratio (pred/ref) {sc['len_ratio']:.2f}")
    return sc

## 10. Time-budgeted training loop (all methods)

In [ ]:
def train_fast(method, train_docs, dev_docs, minutes=45, bs=1, accum=16, lr_bb=5e-5, lr_head=1e-2, teacher=None,
               kd_T=2.0, kd_alpha=0.5, select=None, top_blocks=None, ckpt=True, seed=0, eval_every=10, dev_n=100,
               gen_n=100, init_model=None, **build_kw):
    """Time-budgeted trainer: trains for `minutes` of wall-clock time on random documents (the learning-rate schedule follows
    elapsed time), prints the dev loss every `eval_every` minutes, then generates summaries for gen_n dev documents and scores them.
    method: full | partial | lora | rslora | lorma | prompt | houlsby | pfeiffer | kd.
    top_blocks=n  -> only the last n of the 8 block passes get gradients (embedding + earlier passes run without gradients; cheaper).
    ckpt=True (default here: the windowed attention copies key/value windows, which is memory-hungry) -> gradient checkpointing.   init_model -> continue training a model in memory.
    KD: loss = alpha * CE(reference summary) + (1-alpha) * T^2 * KL(teacher || student) over summary tokens.
    lr_head is the learning rate of the soft prompt (prompt tuning); lr_bb that of every other trainable weight."""
    random.seed(seed); torch.manual_seed(seed)
    if init_model is not None:
        model = init_model
    elif method == "kd":
        assert teacher is not None, "pass teacher=<your fully fine-tuned model>"
        teacher.eval()
        for p in teacher.parameters(): p.requires_grad = False
        model = make_student(teacher).to(dev)
    else:
        model = build_model(method, **build_kw)
    model.use_ckpt = ckpt
    if top_blocks is not None: model.set_top_blocks(top_blocks)
    print(f"[{method}/{select or SELECT}/top={top_blocks}] params {n_params(model)/1e6:.1f}M | trainable {n_params(model, True)/1e6:.2f}M | budget {minutes} min")

    head_ids = {id(p) for p in model.head_params()}
    bb = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    hd = [p for p in model.parameters() if p.requires_grad and id(p) in head_ids]
    groups, base_lrs = [], []
    if bb: groups.append({"params": bb, "lr": lr_bb}); base_lrs.append(lr_bb)
    if hd: groups.append({"params": hd, "lr": lr_head}); base_lrs.append(lr_head)
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")

    budget, t0, mb, seen, run, nrun = minutes * 60.0, time.time(), 0, 0, 0.0, 0
    last_eval = last_log = time.time(); done = False
    while not done:
        order = list(range(len(train_docs))); random.shuffle(order)
        for i in range(0, len(order), bs):
            el = time.time() - t0
            if el >= budget:
                done = True; break
            model.train()
            batch = [train_docs[j] for j in order[i:i + bs]]
            X, L = collate([(get_prompt(d, select), summary_ids(d)) for d in batch])
            with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
                loss, ce = seq_loss(model, X.to(dev), L.to(dev), teacher, kd_T, kd_alpha)
            scaler.scale(loss / accum).backward()
            run += ce.item(); nrun += 1; seen += len(batch); mb += 1
            if mb % accum == 0:
                prog = el / budget
                mult = prog / 0.05 if prog < 0.05 else 0.5 * (1 + math.cos(math.pi * (prog - 0.05) / 0.95))
                for g, b0 in zip(opt.param_groups, base_lrs): g["lr"] = b0 * max(mult, 0.02)
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            if eval_every and time.time() - last_eval > eval_every * 60:
                te = time.time()
                dl = dev_loss(model, dev_docs, dev_n, select=select)
                print(f"   -- {(te-t0)/60:5.1f} min | train CE {run/max(1,nrun):.3f} | dev CE {dl:.3f} (perplexity {math.exp(dl):.1f}) "
                      f"| {seen} docs ({seen/len(train_docs):.2f} epochs)")
                t0 += time.time() - te; last_eval = last_log = time.time(); run, nrun = 0.0, 0
            elif time.time() - last_log > 300:
                print(f"   {(time.time()-t0)/60:5.1f} min | docs seen {seen} ({seen/len(train_docs):.2f} epochs) | {seen/(time.time()-t0):.2f} docs/s | train CE {run/max(1,nrun):.3f}")
                run, nrun, last_log = 0.0, 0, time.time()
    print(f"trained {(time.time()-t0)/60:.1f} min: {seen} docs = {seen/len(train_docs):.2f} epochs ({seen/(time.time()-t0):.2f} docs/s)")
    sc = dev_eval(model, dev_docs, gen_n, select) if dev_docs else {}
    return model, sc

## 11. Submission writer
`prediction.json` (a list of `{"id", "summary"}`) goes at the **root** of the zip. Upload each zip to Codabench.

In [ ]:
def n_params(m, trainable=False):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))

def make_submission(model, docs, tag, select=None, hier=False, **gen_kw):
    """Generate a summary for every test document. prediction.json = [{"id": ..., "summary": ...}, ...] (list format, as the course
    recommends) at the root of the zip. The generations are also saved as JSON for your error analysis."""
    t0 = time.time()
    summ = summarise_hier(model, docs, **gen_kw) if hier else summarise(model, docs, select, **gen_kw)
    sub = [{"id": d["id"], "summary": s} for d, s in zip(docs, summ)]
    assert len(sub) == len(docs) and all(isinstance(s["summary"], str) and s["summary"] for s in sub)
    with open(f"{OUT}/prediction.json", "w", encoding="utf-8") as f:
        json.dump(sub, f, ensure_ascii=False)
    name = f"{OUT}/submission_p6_{tag}.zip"
    with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")
    print(name, "| docs:", len(sub), "| mean words:", round(float(np.mean([len(s['summary'].split()) for s in sub])), 1),
          f"| generation {time.time()-t0:.0f}s")
    return name

results = {}

## 12. Experiment 1 - full fine-tuning (also the KD teacher)
All weights trainable (lr 5e-5) for a fixed 90-minute budget. The cell then checks that the cached decoder matches the plain forward pass - run it once. Keep this model in memory for distillation. If you still run out of GPU memory, restart the kernel, lower `GEN_MAX` in the config, or pass `top_blocks=4`.

In [ ]:
m_full, results["full"] = train_fast("full", train_docs, dev_docs, minutes=90)
check_cache(m_full)
torch.save(m_full.state_dict(), f"{OUT}/p6_lightweight_full.pt")
make_submission(m_full, test_docs, "lightweight_full")

## 13. Experiment 2 - partial (layer-wise) fine-tuning
The embedding and the first repeat (4 of the 8 block passes) run without gradients; the second repeat and the shared block weights it uses train.

In [ ]:
m_part, results["partial"] = train_fast("partial", train_docs, dev_docs, minutes=45)
make_submission(m_part, test_docs, "lightweight_partial")

## 14. Experiment 3 - LoRA
Frozen backbone, rank-16 adapters on every Linear layer of the unique blocks (attention and SwiGLU).

In [ ]:
m_lora, results["lora"] = train_fast("lora", train_docs, dev_docs, minutes=45, lr_bb=5e-4)
make_submission(m_lora, test_docs, "lightweight_lora")

## 15. Experiment 4 - LoRMA+ (compulsory)
Multiplicative low-rank adaptation (Bihany, Patel, Modi 2025) of the query/value projections of the attention in each unique block.

In [ ]:
m_lorma, results["lorma"] = train_fast("lorma", train_docs, dev_docs, minutes=45, lr_bb=3e-4)
make_submission(m_lorma, test_docs, "lightweight_lorma")

## 16. Experiment 5 - Houlsby adapters
A zero-initialised bottleneck adapter after the attention AND after the SwiGLU FFN of every unique block; the backbone stays frozen.

In [ ]:
m_houl, results["houlsby"] = train_fast("houlsby", train_docs, dev_docs, minutes=45, lr_bb=1e-3)
make_submission(m_houl, test_docs, "lightweight_houlsby")

## 17. Experiment 6 - Pfeiffer adapters
One adapter per block, after the FFN only (half the adapter parameters of Houlsby).

In [ ]:
m_pf, results["pfeiffer"] = train_fast("pfeiffer", train_docs, dev_docs, minutes=45, lr_bb=1e-3)
make_submission(m_pf, test_docs, "lightweight_pfeiffer")

## 18. Experiment 7 - prompt tuning
Only 20 learned soft-prompt vectors prepended to the input train; the whole model is frozen. (Often weak for generation - that is a result.)

In [ ]:
m_prompt, results["prompt"] = train_fast("prompt", train_docs, dev_docs, minutes=45, lr_head=3e-2)
make_submission(m_prompt, test_docs, "lightweight_prompt")

## 19. Experiment 8 - knowledge distillation
Teacher = experiment 1. Student = teacher keeping unique blocks 0 and 2 (half the block parameters; effective depth 8 -> 4), initialised from the teacher, trained on the reference summaries + the teacher's temperature-scaled token distributions. Compare its parameter count, generation speed and ROUGE with the teacher.

In [ ]:
m_kd, results["kd"] = train_fast("kd", train_docs, dev_docs, minutes=45, teacher=m_full, lr_bb=5e-5)
make_submission(m_kd, test_docs, "lightweight_kd")
print("params: teacher", round(n_params(m_full)/1e6, 1), "M | student", round(n_params(m_kd)/1e6, 1), "M")
t0 = time.time(); summarise(m_full, dev_docs[:32]); tt = time.time() - t0
t0 = time.time(); summarise(m_kd, dev_docs[:32]); ts = time.time() - t0
print(f"generation time for 32 dev docs: teacher {tt:.0f}s | student {ts:.0f}s")

## 20. Long-document ablation (inference only, uses experiment 1)
Which part of the document carries the summary information? The same model is prompted with the beginning only, beginning + end, tf-idf-selected sentences, and the hierarchical chunk-then-summarise procedure. (For a fair comparison of strategies, retrain with the strategy via `train_fast(..., select=...)`.)

In [ ]:
for sel in ["head", "headtail", "importance"]:
    print(sel); results[f"sel_{sel}"] = dev_eval(m_full, dev_docs, 100, select=sel)
print("hierarchical")
sub = dev_docs[:100]; preds = summarise_hier(m_full, sub)
results["sel_hier"] = summary_scores(preds, [d["summary"] for d in sub]); print({k: round(v, 4) for k, v in results["sel_hier"].items()})

## 21. Generation-length and redundancy ablation (inference only)
Effect of the minimum / maximum summary length and of the no-repeat n-gram rule on ROUGE, repetition and the length ratio.

In [ ]:
for kw in [dict(nrep=0, min_new=30), dict(nrep=3, min_new=30), dict(nrep=4, min_new=30),
           dict(nrep=3, min_new=60), dict(nrep=3, min_new=30, max_new=100), dict(nrep=3, min_new=30, max_new=GEN_MAX)]:
    print(kw); results[f"gen_{kw}"] = dev_eval(m_full, dev_docs, 100, **kw)

## 22. (Optional) BERTScore on dev
BERTScore is the primary metric but needs a pretrained encoder. The course forbids pretrained models for **training** the systems; computing a metric is a different use, but **check with the TAs** before relying on it. Off by default; the numbers will not match Codabench exactly.

In [ ]:
RUN_BERTSCORE = False
if RUN_BERTSCORE:
    # !pip install bert-score
    from bert_score import score as bs_score
    sub = dev_docs[:100]; preds = summarise(m_full, sub)
    P_, R_, F1_ = bs_score(preds, [d["summary"] for d in sub], lang="en", verbose=False)
    print("BERTScore F1:", F1_.mean().item())

## 23. Summary and example generations

In [ ]:
print("dev results (first 100 dev docs):")
for k, v in results.items():
    if isinstance(v, dict) and "rougeL" in v: print(f"  {k:46s} R1 {v['rouge1']:.4f} R2 {v['rouge2']:.4f} RL {v['rougeL']:.4f} | rep {v['rep']:.3f} | len ratio {v['len_ratio']:.2f}")

sc = dev_eval(m_full, dev_docs, 5, verbose=False)
for d, p in list(zip(dev_docs, sc["preds"]))[:3]:
    print("\nREFERENCE:", d["summary"][:500]); print("GENERATED:", p[:500])

## 24. (Optional) retrain on train + dev
Once you know the best method, budget and generation settings, retrain on all labelled data and submit again.
```python
m_final, _ = train_fast("full", train_docs + dev_docs, [], minutes=<same as the best run>)
make_submission(m_final, test_docs, "lightweight_final")
```